# Multi-Model Agricultural Production Forecasting (Optimized)
This notebook trains and compares both **Global Machine Learning Models** (Linear Regression, Random Forest, XGBoost) and **Local Time Series Models** (ARIMA, SARIMA). It has been specifically tuned with heavy regularization to eliminate overfitting and ensure generalization.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX


## 2. Load and Preprocess Data

In [ ]:
try:
    df = pd.read_csv('monthly_agricultural_data.csv')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("ERROR: Please upload 'monthly_agricultural_data.csv' to Colab.")
    df = pd.DataFrame()

if not df.empty:
    df = df.dropna(subset=['Productio'])
    df = df[df['Productio'] >= 0]

    MONTH_MAP = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4, 'May': 5, 'June': 6,
        'July': 7, 'August': 8, 'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    if 'month_num' not in df.columns:
        df['month_num'] = df['month'].map(MONTH_MAP)

    df['date'] = pd.to_datetime(df['year'].astype(str) + '-' + df['month_num'].astype(str) + '-01')
    df['month_sin'] = np.sin(2 * np.pi * df['month_num']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month_num']/12)
    
    df = df.sort_values(by=['Crop_nam', 'Location_district', 'date'])

    for lag in [1, 2, 3, 6, 12]:
        df[f'Productio_lag_{lag}'] = df.groupby(['Crop_nam', 'Location_district'])['Productio'].shift(lag)
    for win in [3, 6, 12]:
        df[f'Productio_rolling_mean_{win}'] = df.groupby(['Crop_nam', 'Location_district'])['Productio'].transform(lambda x: x.rolling(win).mean())

    df['harvested'] = df['harvested'].fillna(0) if 'harvested' in df.columns else 0
    df['yield'] = df['yield'].fillna(0) if 'yield' in df.columns else 0

    df = df.dropna(subset=['Productio'])
    df = df.sort_values('date')


## 3. Global ML Model Training (Regularized for Generalization)

In [ ]:
ml_results = {}

if not df.empty:
    feature_cols = ['year', 'month_sin', 'month_cos',
                    'Productio_lag_1', 'Productio_lag_2', 'Productio_lag_3',
                    'Productio_lag_6', 'Productio_lag_12',
                    'Productio_rolling_mean_3', 'Productio_rolling_mean_6', 'Productio_rolling_mean_12',
                    'harvested', 'yield', 'Crop_nam', 'Location_district']
    
    le_crop, le_dist = LabelEncoder(), LabelEncoder()
    df['Crop_enc'] = le_crop.fit_transform(df['Crop_nam'])
    df['Dist_enc'] = le_dist.fit_transform(df['Location_district'])
    
    feature_cols_encoded = [c for c in feature_cols if c not in ['Crop_nam', 'Location_district']] + ['Crop_enc', 'Dist_enc']
    
    train_df = df[df['date'] < '2020-01-01']
    val_df = df[(df['date'] >= '2020-01-01') & (df['date'] < '2022-01-01')]
    test_df = df[df['date'] >= '2022-01-01']
    
    X_train, y_train = train_df[feature_cols_encoded].fillna(0), train_df['Productio']
    X_val, y_val = val_df[feature_cols_encoded].fillna(0), val_df['Productio']
    X_test, y_test = test_df[feature_cols_encoded].fillna(0), test_df['Productio']
    
    scaler = StandardScaler()
    numeric_cols = ['year', 'month_sin', 'month_cos', 'harvested', 'yield']
    X_train.loc[:, numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
    X_val.loc[:, numeric_cols] = scaler.transform(X_val[numeric_cols])
    X_test.loc[:, numeric_cols] = scaler.transform(X_test[numeric_cols])
    
    # 1. Linear Regression
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)
    ml_results['Linear Regression'] = mean_absolute_error(y_test, lr_model.predict(X_test))
    
    # 2. Random Forest (Tuned to resolve overfit)
    rf_model = RandomForestRegressor(
        n_estimators=100, 
        max_depth=5, 
        min_samples_split=10, 
        min_samples_leaf=5, 
        max_features='sqrt', 
        random_state=42, 
        n_jobs=-1
    )
    rf_model.fit(X_train, y_train)
    ml_results['Random Forest'] = mean_absolute_error(y_test, rf_model.predict(X_test))
    
    # 3. XGBoost (Tuned to resolve overfit)
    xgb_model = xgb.XGBRegressor(
        n_estimators=1000, 
        max_depth=3, 
        learning_rate=0.01, 
        gamma=5, 
        reg_lambda=5.0, 
        random_state=42
    )
    xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    ml_results['XGBoost'] = mean_absolute_error(y_test, xgb_model.predict(X_test))


## 4. Local Time Series Model Training (Simplified for Generalization)

In [ ]:
ts_results = {}
if not df.empty:
    ts_crop, ts_dist = 'Cabbage', 'Anuradhapura'
    ts_df = df[(df['Crop_nam'] == ts_crop) & (df['Location_district'] == ts_dist)]
    ts_df = ts_df.set_index('date').asfreq('MS')
    ts_df['Productio'] = ts_df['Productio'].ffill()

    ts_train = ts_df[ts_df.index < '2022-01-01']['Productio']
    ts_test = ts_df[ts_df.index >= '2022-01-01']['Productio']

    if len(ts_train) > 0 and len(ts_test) > 0:
        # ARIMA (Simplified)
        arima = ARIMA(ts_train, order=(1,1,1)).fit()
        arima_preds = arima.forecast(steps=len(ts_test))
        ts_results['ARIMA'] = mean_absolute_error(ts_test, arima_preds)
        
        # SARIMA (Simplified)
        sarima = SARIMAX(ts_train, order=(0, 1, 1), seasonal_order=(0, 1, 0, 12)).fit(disp=False)
        sarima_preds = sarima.forecast(steps=len(ts_test))
        ts_results['SARIMA'] = mean_absolute_error(ts_test, sarima_preds)


## 5. Comparative Evaluation Graph

In [ ]:
if not df.empty:
    all_results = {**ml_results, **ts_results}
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(all_results.keys(), all_results.values(), color=['skyblue', 'lightgreen', 'salmon', 'plum', 'tan'])
    plt.ylabel('Mean Absolute Error (Test Set)')
    plt.title('Final Benchmarking: All Models Optimized for Generalization')
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}", ha='center', va='bottom')
    plt.show()
